# 03 · Competição entre os dois algoritmos

**Modelo A — Regressão Logística** e **Modelo B — HistGradientBoosting** (o algoritmo do modelo da interface).
Os dois usam **as mesmas features eleitas** no notebook 02, **o mesmo treino**, **a mesma validação** (6 meses
finais do ciclo) e **a mesma métrica principal** (PR-AUC). Cada um é um `Pipeline` com o pré-processamento
(imputação, escala, codificação) ajustado só no treino, e é registrado no MLflow com o `cycle_id`.

In [1]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
import joblib
from src import modelagem as md, preprocessamento as pp
ciclo = ler_ciclo(cfg)
treino = pd.read_parquet(estado(cfg, "processados") / "treino.parquet")
validacao = pd.read_parquet(estado(cfg, "processados") / "validacao.parquet")
eleitas, seed = ciclo["features_eleitas"], cfg["modelos"]["seed"]
y_val = validacao[pp.ALVO].to_numpy()
print(ciclo["cycle_id"], "| features:", eleitas)

20261124-2008-45477532512 | features: ['taxa_historica_periodo', 'acid_ult_365d', 'lon', 'delegacia', 'br', 'taxa_historica', 'periodo_idx', 'lat', 'vmd', 'nebulosidade']


In [3]:
mlflow = configurar_mlflow(cfg)
pasta_cand = estado(cfg, "processados") / "candidatos"; pasta_cand.mkdir(parents=True, exist_ok=True)
candidatos = {}
for papel in ["A", "B"]:
    alg = cfg["modelos"][papel]
    modelo = md.treinar(alg, treino, eleitas, seed)
    m = md.metricas(y_val, md.prever(modelo, validacao, eleitas))
    with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · modelo {papel} ({alg})",
                          tags=tags_ciclo(ciclo, etapa="treino", papel="candidato", modelo=papel, algoritmo=alg)) as run:
        params = {k: v for k, v in modelo.named_steps["modelo"].get_params().items() if isinstance(v, (int, float, str, bool, type(None)))}
        mlflow.log_params({**params, "features": ",".join(eleitas), "n_features": len(eleitas), "seed": seed,
                           "linhas_treino": len(treino)})
        mlflow.log_metrics({f"val_{k}": v for k, v in m.items()})
        mlflow.sklearn.log_model(modelo, name="modelo", serialization_format="cloudpickle")  # modelo gerado aqui; mesmo formato do joblib
    joblib.dump(modelo, pasta_cand / f"{alg}.joblib")
    candidatos[alg] = {"modelo": papel, "algoritmo": alg, "run_id": run.info.run_id, "metricas": m}
comparacao = pd.DataFrame({a: c["metricas"] for a, c in candidatos.items()}).T
comparacao

2026/11/24 20:09:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/24 20:10:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,roc_auc,pr_auc,brier,razao_prevista_observada,captura_top_10pct,captura_top_5pct
logistica,0.8010,0.1264,0.01870,0.992,0.4763,0.3517
hist_gradient_boosting,0.8199,0.1334,0.01856,0.971,0.5003,0.3652


In [4]:
comparacao.to_csv(pasta_evidencias(cfg, ciclo) / "comparacao.csv")
ciclo = atualizar_ciclo(cfg, candidatos=candidatos)